In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import os

In [3]:
tickers = ["GOOGL", "JPM", "TSLA", "XOM", "PFE"]
save_raw_path = "/Users/lohitha._.kalepu/Documents/DS492"
os.makedirs(save_raw_path, exist_ok=True)

raw_by_ticker = {}

for t in tickers:
    df = yf.download(t, start="2005-01-01", end="2026-01-01", auto_adjust=False, interval="1d")

    if df.empty:
        print("  failed/empty:", t)
        continue

    df = df.reset_index()
    df["Ticker"] = t
    raw_by_ticker[t] = df
    df.to_parquet(os.path.join(save_raw_path, f"{t}_raw.parquet"), index=False)

print("tickers saved:", list(raw_by_ticker.keys()))

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

tickers saved: ['GOOGL', 'JPM', 'TSLA', 'XOM', 'PFE']


In [4]:
folder = "/Users/lohitha._.kalepu/Documents/DS492"

dfs = []
for t in ["GOOGL", "JPM", "TSLA", "XOM", "PFE"]:
    d = pd.read_parquet(os.path.join(folder, f"{t}_raw.parquet"))

    # ensure Date column exists
    if "Date" not in d.columns:
        d = d.reset_index().rename(columns={"index": "Date"})

    # ensure Ticker exists
    if "Ticker" not in d.columns:
        d["Ticker"] = t

    dfs.append(d)

all_raw = pd.concat(dfs, ignore_index=True)
all_raw["Date"] = pd.to_datetime(all_raw["Date"])
all_raw = all_raw.sort_values(["Ticker", "Date"]).reset_index(drop=True)

all_raw.to_parquet(os.path.join(folder, "ALL_raw_long.parquet"), index=False)

print(all_raw["Ticker"].value_counts())
print(all_raw.groupby("Ticker")["Date"].agg(["min","max","count"]))

Ticker
GOOGL    5283
JPM      5283
PFE      5283
XOM      5283
TSLA     3902
Name: count, dtype: int64
              min                           max  count
Ticker                                                
GOOGL  1970-01-01 1970-01-01 00:00:00.000005282   5283
JPM    1970-01-01 1970-01-01 00:00:00.000005282   5283
PFE    1970-01-01 1970-01-01 00:00:00.000005282   5283
TSLA   1970-01-01 1970-01-01 00:00:00.000003901   3902
XOM    1970-01-01 1970-01-01 00:00:00.000005282   5283
